In [1]:
import copy
import os
from tqdm import tqdm
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from src.utils.generate_preferences import generate as generate_preferences
from src.utils.generate_prompts import generate as generate_prompts
from src.utils.generate_responses import generate as generate_responses
from src.utils.generate_responses import do_sample
from src.utils.generate_scores import generate as generate_scores
from src.utils.read_write_jsonl import read_jsonl_file, write_jsonl_file
from src.utils.generate_dpo_dataset import generate_dpo_dataset
from utils.FedSPA_utils import compute_ranknet_loss
# from unsloth import FastLanguageModel
import pandas as pd
from vllm import LLM, SamplingParams
import gc
import torch

2024-12-18 20:33:33,514 - datasets - INFO - PyTorch version 2.5.1 available.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken?
A possible explanation is you have a new CUDA version which isn't
yet compatible with FA2? Please file a ticket to Unsloth or FA2.
We shall now use Xformers instead, which does not have any performance hits!
We found this negligible impact by benchmarking on 1x A100.
🦥 Unsloth Zoo will now patch everything to make training faster!
[2024-12-18 20:33:35,415] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)
2024-12-18 20:33:35,482 - root - INFO - gcc -pthread -B /home/tiger/miniconda3/envs/llm/compiler_compat -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /home/tiger/miniconda3/envs/llm/include -fPIC -O2 -isystem /home/tiger/miniconda3/envs/llm/include -fPIC -c /tmp/tmpqmi4mhsl/test.c -o /tmp/tmpqmi4mhsl/test.o
2024-12-18 20:33:35,499

/home/tiger/miniconda3/envs/llm/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/tiger/miniconda3/envs/llm/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/tiger/miniconda3/envs/llm/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/tiger/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/tiger/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/tiger/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/tiger/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/lib

In [2]:
model = AutoModelForCausalLM.from_pretrained(
    "/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/M0_merged",
    torch_dtype=torch.bfloat16,
).to("cuda")

tokenizer = AutoTokenizer.from_pretrained("/mnt/bn/merlin-datavolume-tsy/leon/checkpoints/self-rewarding/M0_merged")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
import datasets
eval_set = datasets.load_dataset("json", data_files="alpaca_eval_output/eval_set_256.json")["train"]

print(eval_set[:3])

{'instruction': ['Design a medium-level sudoku puzzle.', "On the basis of the subject of the email, determine whether the email should be treated as spam or not.\n\nDermatologists don't like her!", 'rank the following companies by how pro-consumer they are:\nMicrosoft, Google, Nintendo, Sony, EA.'], 'output': ['2   6   7\n9   4   1\n3   5   8\n\n1   8   3\n4   2   6\n7   9   5\n\n8   7   5\n6   3   4\n9   1   2\n\n5   1   6\n2   8   9\n4   7   3', 'Not Spam', 'Google, Microsoft, Nintendo, Sony, EA.'], 'generator': ['text_davinci_003', 'text_davinci_003', 'text_davinci_003'], 'dataset': ['selfinstruct', 'selfinstruct', 'koala']}


In [4]:
eval_set = eval_set.to_list()

In [5]:
do_sample(model, tokenizer, eval_set[0]["instruction"])

"Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nDesign a medium-level sudoku puzzle. \n\n### Response:Here's a medium-level sudoku puzzle:\n\n1 2 3 4 5 6 7 8 9\n4 5 6 7 8 9 1 2 3\n7 8 9 1 2 3 4 5 6\n2 3 1 5 6 4 8 9 7\n5 6 4 8 9 7 2 3 1\n8 9 7 2 3 1 5 6 4\n3 1 2 6 4 5 7 8 9\n6 4 5 3 1 2 9 7 8\n9 7 8 3 1 2 6 4 5\n\nYou can verify your solution here: https://www.sudoku-solver.net/solve"

In [6]:
ref_model = AutoModelForCausalLM.from_pretrained(
    "/mnt/bn/data-tns-live-llm/leon/datasets/Meta-Llama-3-8B",
    torch_dtype=torch.bfloat16,
).to("cuda")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nDesign a medium-level sudoku puzzle. \n\n### Response:"""
completion = """Here's a medium-level sudoku puzzle that I designed:\n\n```\n4 6 7 | 8 9 1 | 2 5 3\n5 3 1 | 2 4 7 | 6 8 9\n9 8 2 | 3 6 5 | 4 1 7\n---+---+---\n6 4 3 | 1 2 9 | 8 7 5\n7 5 8 | 4 3 6 | 9 2 1\n2 1 9 | 5 7 8 | 3 6 4\n---+---+---\n3 9 5 | 7 1 4 | 2 6 8\n1 2 6 | 9 8 3 | 5 4 7\n8 7 4 | 6 5 2 | 1 9 3\n```\n\nTo solve this puzzle, you can use the following rules:\n\n1. Each row, column, and 3x3 box must contain the numbers 1-9, and each number can only appear once in a row, column, or box.\n2"""

In [10]:
from transformers import PreTrainedTokenizer, PreTrainedModel
def compute_sequence_logprobs(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    prompt: str,
    response: str,
) -> float:
    """
    计算给定响应的条件对数概率。使用累积的token级别对数概率。

    Args:
        model: 预训练模型
        tokenizer: 分词器
        prompt: 输入提示
        response: 模型响应

    Returns:
        响应序列的累积对数概率
    """
    try:
        # 构建输入
        full_prompt = prompt
        
        # 对prompt和response分别编码
        prompt_ids = tokenizer(full_prompt, return_tensors="pt")["input_ids"].to(model.device)
        full_input = tokenizer(full_prompt + response, return_tensors="pt")
        full_input_ids = full_input["input_ids"].to(model.device)
        
        # 构建labels，prompt部分用-100屏蔽
        labels = torch.full_like(full_input_ids, -100)
        prompt_len = prompt_ids.shape[1]
        labels[:, prompt_len:] = full_input_ids[:, prompt_len:]
        
        # 获取模型输出
        with torch.no_grad():
            outputs = model(full_input_ids)
            logits = outputs.logits[:, :-1, :]  # [batch, seq_len, vocab_size]
            labels = labels[:, 1:]  # 移除第一个token，因为我们shift了logits
            
            # 计算log softmax
            log_probs = torch.log_softmax(logits, dim=-1)
            
            # 创建mask，只关注response部分
            labels_mask = (labels != -100)
            
            # 获取每个位置的实际token的log prob
            # 确保labels中的-100被替换为0，避免gather时的索引越界
            valid_labels = labels.clone()
            valid_labels[valid_labels == -100] = 0
            
            token_log_probs = torch.gather(
                log_probs, 
                dim=2, 
                index=valid_labels.unsqueeze(2)
            ).squeeze(2)
            
            # 只统计response部分的log prob总和
            print(token_log_probs)
            print(labels_mask)
            sequence_log_prob = (token_log_probs * labels_mask).sum()
            
            return sequence_log_prob.item()
        
    except Exception as e:
        print(f"Error computing sequence logprobs: {e}")
        return float("-inf")

In [11]:
current_logprob = compute_sequence_logprobs(
    model, tokenizer, prompt, completion
)
ref_logprob = compute_sequence_logprobs(
    ref_model, tokenizer, prompt, completion
)

# 计算内在奖励 r_θ(x,y) ∝ [log_π_θ(y|x) - log_π_ref(y|x)]
intrinsic_reward = current_logprob - ref_logprob

print(current_logprob)
print(ref_logprob)
print(intrinsic_reward)

tensor([[-8.7500e+00, -1.2188e+01, -1.8000e+01, -1.6000e+01, -1.1688e+01,
         -1.5188e+01, -1.4812e+01, -1.4875e+01, -1.0000e+01, -1.8375e+01,
         -1.2375e+01, -1.5188e+01, -1.0000e+01, -1.4625e+01, -1.4312e+01,
         -1.5875e+01, -1.8375e+01, -8.4375e+00, -7.7500e+00, -1.2250e+01,
         -1.3625e+01, -8.3750e+00, -1.2750e+01, -1.5812e+01, -1.2000e+01,
         -1.3438e+01, -9.5625e+00, -6.6562e+00, -1.5750e+01, -8.8125e+00,
         -1.8250e+01, -1.5125e+01, -5.8203e-01, -8.4375e-01, -1.1230e-01,
         -4.5117e-01, -1.1377e-01, -7.8125e-01, -1.5564e-02, -2.0312e+00,
         -2.1875e+00, -1.1250e+00, -8.0859e-01, -1.3438e+00, -9.5215e-03,
         -1.9375e+00, -6.2891e-01, -1.9922e+00, -2.6562e-01, -1.6328e+00,
         -3.8867e-01, -2.1973e-02, -1.9375e+00, -7.0801e-02, -1.8359e+00,
         -4.2480e-02, -1.1250e+00, -5.8899e-03, -4.2725e-03, -5.0781e-01,
         -4.3945e-03, -1.1484e+00, -8.6060e-03, -2.8381e-03, -1.6797e-01,
         -1.5781e+00, -4.9133e-03, -9.